# Claude Agent SDK: structured output (validated JSON from agents)

Agents answer in free-form text by default. That is fine for a chat, and a problem the moment your **code** needs the answer: you would have to parse "15 minutes" into a number, split ingredients from steps, and cope with different formatting every run.

**Structured output** lets you define the **exact shape** of the data you want. The agent still uses whatever tools it needs (search, read files, run commands), and at the end you get **validated JSON that matches your schema**. If the output does not validate, the SDK **re-prompts**; if it still fails within the retry limit, you get an **error result** instead of bad data.

**What you will learn (and see running):**
1. Describing your data with **JSON Schema**
2. Getting validated output with `output_format`
3. Typed objects with **Pydantic**
4. Multi-step tool use feeding a structured answer (a TODO tracker)
5. **Error handling**: every way a run can end, and how to force and handle failures
6. Tips for schemas that actually succeed

## The idea in one picture

```
  your JSON Schema ──┐
                     ▼
  prompt ──▶ agent loop (uses tools as needed) ──▶ proposes JSON
                                                     │
                              validate against schema (JSON Schema draft-07)
                                  ├── valid ───────▶ ResultMessage.structured_output
                                  └── invalid ──▶ re-prompt … up to a retry limit
                                                     └──▶ error_max_structured_output_retries
```

| Without structured output | With structured output |
|---|---|
| `"Prep time: 15 minutes | Cook time: 10 minutes ..."` | `{"prep_time_minutes": 15, "cook_time_minutes": 10, ...}` |
| You parse and hope | You use typed data directly |

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv pydantic jsonschema

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("structured-output-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Some local data to work on

We keep everything local (no web needed): a contacts file and a tiny codebase with `TODO` comments.

In [ ]:
(DEMO_DIR / "contacts.txt").write_text(
    "Meeting notes, 12 March.\n"
    "Please follow up with Dana Whitfield, VP of Engineering at Northwind Labs.\n"
    "Her office line is 555-0142. No email address was shared.\n"
)

src = DEMO_DIR / "src"
src.mkdir(exist_ok=True)
(src / "app.py").write_text(
    "def handler(request):\n"
    "    # TODO: add caching for repeated requests\n"
    "    return process(request)\n\n"
    "def process(request):\n"
    "    # TODO: validate the request body\n"
    "    return {}\n"
)
(src / "utils.py").write_text(
    "def average(values):\n"
    "    # TODO: handle an empty list\n"
    "    return sum(values) / len(values)\n"
)
print([str(p.relative_to(DEMO_DIR)) for p in sorted(DEMO_DIR.rglob("*")) if p.is_file()])

In [ ]:
# OFFLINE
import json
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

def short(value, n=160):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"

## 2. Describe the shape with JSON Schema

A schema says what keys exist, their types, and which are **required**. The SDK supports standard JSON Schema: all basic types (`object`, `array`, `string`, `number`, `boolean`, `null`), `enum`, `const`, `required`, nested objects and `$ref` definitions. It validates with **draft-07**.

Before involving the agent, let's see what *validation* means by trying the schema on hand-written data. The `jsonschema` library implements the same standard.

In [ ]:
# OFFLINE
from jsonschema import Draft7Validator

company_schema = {
    "type": "object",
    "properties": {
        "company_name": {"type": "string"},
        "founded_year": {"type": "number"},
        "headquarters": {"type": "string"},
    },
    "required": ["company_name"],
}

Draft7Validator.check_schema(company_schema)      # the schema itself is valid
validator = Draft7Validator(company_schema)

samples = {
    "complete":         {"company_name": "Acme", "founded_year": 1999, "headquarters": "Oslo"},
    "only required":    {"company_name": "Acme"},
    "missing required": {"founded_year": 1999},
    "wrong type":       {"company_name": "Acme", "founded_year": "nineteen ninety-nine"},
}
for name, data in samples.items():
    errors = [e.message for e in validator.iter_errors(data)]
    print(f"{name:17} -> {'VALID' if not errors else 'INVALID: ' + errors[0]}")

Two details worth knowing:

- A schema that is **not valid JSON Schema fails the run at startup** with an error naming the problem (older Claude Code versions silently ignored it).
- The **`format`** keyword (`"format": "email"`) is accepted as an **annotation** and **not enforced**. A string that is not an email still passes. The standard validator behaves the same by default:

In [ ]:
# OFFLINE
email_schema = {"type": "object", "properties": {"email": {"type": "string", "format": "email"}}}
print("not-an-email passes?:", Draft7Validator(email_schema).is_valid({"email": "not-an-email"}))

## 3. Quick start: get validated JSON

Pass the schema in `output_format`:

```python
ClaudeAgentOptions(output_format={"type": "json_schema", "schema": schema})
```

When the agent finishes, the `ResultMessage` carries a **`structured_output`** field with the validated data. (The answer text may also be in `result`, but **`structured_output`** is what you use.)

First, a reusable helper that implements the **complete success check** the docs recommend: success means `subtype == "success"` **and** `structured_output` is present. Everything else is a failure.

In [ ]:
async def get_structured(prompt, schema, **opts):
    """Run the agent and return validated structured data (or None). Prints how the run ended."""
    opts.setdefault("cwd", str(DEMO_DIR))
    data = None
    try:
        async for message in query(
            prompt=prompt,
            options=ClaudeAgentOptions(output_format={"type": "json_schema", "schema": schema}, **opts),
        ):
            if isinstance(message, ResultMessage):
                if message.subtype == "success" and message.structured_output:
                    data = message.structured_output
                    status = "OK"
                elif message.subtype == "error_max_structured_output_retries":
                    status = "FAILED: no valid output after several attempts"
                    if message.errors:
                        print("   errors:", short(message.errors, 300))
                else:
                    status = f"FAILED: run ended without structured output ({message.subtype})"
                cost = f"${message.total_cost_usd:.4f}" if message.total_cost_usd is not None else "n/a"
                print(f"   [{status}] turns={message.num_turns} cost={cost}")
    except Exception as error:
        print("   (query raised:", short(error, 200), ")")
    return data

In [ ]:
contact_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "title": {"type": "string"},
        "company": {"type": "string"},
        "phone": {"type": "string"},
        "email": {"type": "string"},
    },
    "required": ["name"],       # everything else is optional
}

contact = await get_structured(
    "Read contacts.txt and extract the contact's details. Leave out anything that is not stated.",
    contact_schema,
    allowed_tools=["Read"],
)
print(json.dumps(contact, indent=2))

Because `email` is **optional**, the agent could omit it honestly instead of inventing one. That design choice (below) is the single most useful tip for schemas that succeed.

## 4. Type-safe schemas with Pydantic

Writing JSON Schema by hand gets old. With **Pydantic** (Python) you define a class and get both the **schema** (`model_json_schema()`) and a **typed object** back (`model_validate(...)`). (In TypeScript the equivalent is Zod: `z.toJSONSchema(schema, { target: "draft-7" })`.)

`Literal[...]` is the neat way to say "one of these strings" (it becomes an `enum`).

In [ ]:
# OFFLINE
from typing import Literal
from pydantic import BaseModel, ValidationError


class Step(BaseModel):
    step_number: int
    description: str
    estimated_complexity: Literal["low", "medium", "high"]


class FeaturePlan(BaseModel):
    feature_name: str
    summary: str
    steps: list[Step]
    risks: list[str]


plan_schema = FeaturePlan.model_json_schema()
Draft7Validator.check_schema(plan_schema)       # SDK expects draft-07-compatible schemas
print("top-level keys:", list(plan_schema))
print("complexity enum:", plan_schema["$defs"]["Step"]["properties"]["estimated_complexity"].get("enum"))

good = {"feature_name": "Dark mode", "summary": "Add a theme toggle",
        "steps": [{"step_number": 1, "description": "Define colour tokens", "estimated_complexity": "low"}],
        "risks": ["Contrast regressions"]}
bad = {**good, "steps": [{"step_number": 1, "description": "x", "estimated_complexity": "extreme"}]}

print("good ->", FeaturePlan.model_validate(good).feature_name)
try:
    FeaturePlan.model_validate(bad)
except ValidationError as e:
    print("bad  -> rejected:", e.errors()[0]["msg"])

In [ ]:
raw = await get_structured(
    "Plan how to add dark mode support to a small web app. Break it into implementation steps. "
    "Do not use any tools.",
    plan_schema,
)

if raw:
    plan = FeaturePlan.model_validate(raw)          # now a fully typed object
    print(f"Feature: {plan.feature_name}")
    print(f"Summary: {plan.summary}")
    for step in plan.steps:
        print(f"  {step.step_number}. [{step.estimated_complexity}] {step.description}")
    print("Risks:", "; ".join(plan.risks))

## 5. Structured output after real tool use: a TODO tracker

Here the agent has to **work**: search the code for `TODO` comments, read the files, and combine everything into one structured list. It decides for itself to use `Grep` and `Read`.

The schema marks `author` and `date` **optional**, because nothing in our files provides them (the real docs example reads `git blame`). The agent fills in what it can and omits the rest.

In [ ]:
todo_schema = {
    "type": "object",
    "properties": {
        "todos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"},
                    "file": {"type": "string"},
                    "line": {"type": "number"},
                    "author": {"type": "string"},
                    "date": {"type": "string"},
                },
                "required": ["text", "file", "line"],
            },
        },
        "total_count": {"type": "number"},
    },
    "required": ["todos", "total_count"],
}

found = await get_structured(
    "Find every TODO comment in the code under src/ and report each one with its file and line number.",
    todo_schema,
    allowed_tools=["Grep", "Read", "Glob"],
)

if found:
    print(f"Found {found['total_count']} TODOs")
    for todo in found["todos"]:
        print(f"  {todo['file']}:{todo['line']} - {todo['text']}")
        if "author" in todo:
            print(f"      added by {todo['author']} on {todo.get('date')}")

**Verify, don't trust.** Validation guarantees the *shape*, not that the *content* is right. Let's count the real TODOs ourselves and compare.

In [ ]:
actual = []
for path in sorted((DEMO_DIR / "src").glob("*.py")):
    for number, line in enumerate(path.read_text().splitlines(), start=1):
        if "TODO" in line:
            actual.append((path.name, number))
print("actual TODOs        :", actual)

if found:
    reported = sorted((t["file"].split("/")[-1], int(t["line"])) for t in found["todos"])
    print("agent reported      :", reported)
    print("same locations?     :", reported == sorted(actual))
    print("count matches?      :", found["total_count"] == len(actual))

## 6. Error handling: every way a run can end

| `subtype` | Meaning |
|---|---|
| `success` | Output generated **and** validated |
| `error_max_structured_output_retries` | No valid output remained after multiple attempts |

Why a run might fail:
- the **schema is too complex** for the task (deeply nested, many required fields)
- the **task is ambiguous** or lacks the information a required field needs
- the agent **hit the retry limit** fixing validation errors
- a **model fallback** retracted an already-completed output mid-stream and no retry replaced it. Look at the `errors` list on the result to tell this apart from a schema problem.

**Also treat this as a failure:** `subtype == "success"` but `structured_output` is empty (the run finished without the agent producing one). Our `get_structured` helper already handles all three cases.

### Experiment A: an invalid schema fails at startup
`"not-a-type"` is not a JSON Schema type.

In [ ]:
bad_schema = {"type": "object", "properties": {"x": {"type": "not-a-type"}}}
result = await get_structured("Return x = 1.", bad_schema)
print("data:", result)

### Experiment B: a valid but *unsatisfiable* schema
The schema below is valid JSON Schema, but **no number can be both ≥ 10 and ≤ 5**. The agent will try, fail validation, retry, and finally give up with `error_max_structured_output_retries`. (This costs a few retries; behavior can vary slightly by version.)

In [ ]:
impossible = {
    "type": "object",
    "properties": {"value": {"type": "integer", "minimum": 10, "maximum": 5}},
    "required": ["value"],
}
Draft7Validator.check_schema(impossible)        # structurally valid...
print("any value valid?", any(Draft7Validator(impossible).is_valid({"value": v}) for v in range(-50, 50)))   # ...but unsatisfiable

result = await get_structured("Return an integer called value.", impossible, max_turns=5)
print("data:", result)

### Experiment C: required fields the data cannot supply
`contacts.txt` has **no email**. What happens if the schema *requires* one? Often the agent either invents a plausible one or fails validation. Compare with Section 3, where `email` was optional.

In [ ]:
strict_schema = {**contact_schema, "required": ["name", "email"]}
strict = await get_structured(
    "Read contacts.txt and extract the contact's details.",
    strict_schema,
    allowed_tools=["Read"],
    max_turns=6,
)
print(json.dumps(strict, indent=2))
if strict:
    print("\nCheck contacts.txt: was that email actually in the file? (Validation can't tell you.)")

## 7. Tips for schemas that succeed

- **Keep schemas focused.** Deep nesting with many required fields is harder to satisfy. Start small, then grow.
- **Match the schema to the task.** If the source might not contain a field, make it **optional**. This avoids both failures and invented values.
- **Write clear prompts.** Say what to extract and what to do when data is missing.
- **Constrain with `enum` / `Literal`** for fixed vocabularies; never depend on `format` being enforced.
- **Validate content yourself** when it matters (counts, ranges, cross-field rules). Schema validation checks shape only.
- Using **streaming output**? Streamed JSON fragments are **unvalidated**. Only `ResultMessage.structured_output` is validated.

## 8. Recap

- `output_format={"type": "json_schema", "schema": ...}` makes the agent finish with **validated JSON** in `ResultMessage.structured_output`.
- Build schemas by hand, or with **Pydantic** (`model_json_schema()` + `model_validate`); Zod in TypeScript.
- The agent can use **any tools** before answering, so structured output works for multi-step jobs.
- Success = `subtype == "success"` **and** `structured_output` present. Otherwise handle `error_max_structured_output_retries` and the empty-success case.
- Optional fields prevent failures **and** hallucinated values.
- Validation proves the **shape**, never the **truth**.

### Try it yourself
1. Add a `severity: Literal["low", "medium", "high"]` field to the TODO schema and let the agent judge each one.
2. Make `get_structured` retry once with a *simplified* schema when it fails.
3. Combine this with sessions: store `session_id`, then ask a follow-up that returns a different schema.
4. Wrap the agent in a function `extract_contact(path) -> Contact` that returns a Pydantic model or raises.